# 01 — Data audit

This notebook establishes what is in the Pump It Up files before any cleaning or feature engineering. It measures data-quality problems and records the decisions they create for the baseline notebook.

The audit does **not** replace missing values, drop columns, encode categories or fit a model. Those choices should follow from the evidence collected here.

## Questions to answer

- Do the four files have the expected rows, columns and matching identifiers?
- Which columns contain explicit nulls, blank strings or likely missing-value sentinels?
- How imbalanced is `status_group`?
- Which numeric columns have suspicious zero values or implausible ranges?
- Which categorical columns are constant, high-cardinality or split into overlapping hierarchies?
- Does the test set contain categories absent from the training set?
- Do dates or coordinates expose quality problems, useful features or leakage risks?
- Which findings require a decision before building the baseline?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_dir() -> Path:
    """Find stage-1-pump-it-up whether Jupyter starts here or at the repo root."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data" / "TrainingSetValues.csv").exists():
            return candidate
        nested = candidate / "stage-1-pump-it-up"
        if (nested / "data" / "TrainingSetValues.csv").exists():
            return nested
    raise FileNotFoundError(
        "Could not find the Pump It Up data directory. "
        "Put the DrivenData CSVs in stage-1-pump-it-up/data/."
    )


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "data"
PROJECT_DIR

## Load the source files

Keep the four source frames separate and merge the training labels only after checking identifier alignment.

In [ ]:
train_values = pd.read_csv(DATA_DIR / "TrainingSetValues.csv")
train_labels = pd.read_csv(DATA_DIR / "TrainingSetLabels.csv")
test_values = pd.read_csv(DATA_DIR / "TestSetValues.csv")
submission_format = pd.read_csv(DATA_DIR / "SubmissionFormat.csv")

train = train_values.merge(
    train_labels,
    on="id",
    how="left",
    validate="one_to_one",
)

datasets = {
    "train_values": train_values,
    "train_labels": train_labels,
    "test_values": test_values,
    "submission_format": submission_format,
}

shape_summary = pd.DataFrame(
    {
        "rows": {name: len(frame) for name, frame in datasets.items()},
        "columns": {name: frame.shape[1] for name, frame in datasets.items()},
        "duplicate_ids": {
            name: int(frame["id"].duplicated().sum()) for name, frame in datasets.items()
        },
    }
)
shape_summary

In [ ]:
alignment_checks = pd.Series(
    {
        "training IDs match labels": set(train_values["id"]) == set(train_labels["id"]),
        "test IDs match submission template": set(test_values["id"]) == set(submission_format["id"]),
        "training labels complete after merge": train["status_group"].notna().all(),
        "predictor columns match train and test": list(train_values.columns) == list(test_values.columns),
        "duplicate training rows excluding ID": train_values.drop(columns="id").duplicated().any(),
    },
    name="passed",
)
alignment_checks.to_frame()

In [ ]:
# A transposed sample makes mixed numeric and categorical fields easier to scan.
train.head(3).T

## Column inventory

Start with data types, explicit missingness and cardinality. A low null count does not prove a column is complete: zero and category labels such as `none` may encode missing information.

In [ ]:
def column_inventory(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for column in frame.columns:
        series = frame[column]
        examples = series.dropna().astype(str).drop_duplicates().head(3).tolist()
        rows.append(
            {
                "column": column,
                "dtype": str(series.dtype),
                "missing": int(series.isna().sum()),
                "missing_pct": series.isna().mean() * 100,
                "unique": int(series.nunique(dropna=True)),
                "top_frequency_pct": series.value_counts(dropna=False, normalize=True).iloc[0] * 100,
                "examples": examples,
            }
        )
    return pd.DataFrame(rows).set_index("column")


inventory = column_inventory(train_values)
inventory.sort_values(["missing_pct", "unique"], ascending=[False, False]).round(2)

In [ ]:
constant_columns = inventory.index[inventory["unique"] <= 1].tolist()
high_cardinality_columns = inventory.index[
    inventory["unique"] > 0.20 * len(train_values)
].tolist()

pd.Series(
    {
        "constant columns": constant_columns,
        "columns unique in more than 20% of rows": high_cardinality_columns,
    },
    name="columns",
).to_frame()

## Feature-by-feature review

A disciplined review still needs one pass through every predictor. Use the same questions each time:

1. What does the feature represent, and could it exist at prediction time?
2. What are its type, range, cardinality and most common values?
3. Do nulls, zeros or labels such as `none` carry a real meaning?
4. Does its distribution differ by `status_group`?
5. Does it overlap with another feature or expose an identifier?
6. What treatment should the baseline test?

The helper below selects one of five review templates: numeric, categorical, binary, date, or identifier/free text. Coordinates and related category hierarchies receive paired checks later in the notebook. Change `FEATURE_TO_REVIEW` and record the conclusion in the feature register. Avoid rendering every chart in one run.

In [ ]:
feature_groups = {
    "identifier or free text": ["id", "wpt_name", "num_private"],
    "money and capacity": ["amount_tsh", "population"],
    "time": ["date_recorded", "construction_year"],
    "geography": [
        "gps_height", "longitude", "latitude", "basin", "subvillage",
        "region", "region_code", "district_code", "lga", "ward",
    ],
    "organisation and permissions": [
        "funder", "installer", "public_meeting", "recorded_by", "permit",
    ],
    "scheme and management": [
        "scheme_management", "scheme_name", "management", "management_group",
    ],
    "extraction": ["extraction_type", "extraction_type_group", "extraction_type_class"],
    "payment": ["payment", "payment_type"],
    "water quality and quantity": ["water_quality", "quality_group", "quantity", "quantity_group"],
    "source": ["source", "source_type", "source_class"],
    "waterpoint": ["waterpoint_type", "waterpoint_type_group"],
}

feature_to_group = {
    feature: group
    for group, features in feature_groups.items()
    for feature in features
}

def classify_feature(feature: str) -> str:
    if feature == "date_recorded":
        return "date"
    if feature in {"id", "wpt_name"}:
        return "identifier/free text"
    if feature in {"public_meeting", "permit"}:
        return "binary"
    if pd.api.types.is_numeric_dtype(train_values[feature]):
        return "numeric"
    return "categorical"


feature_register = inventory.reset_index().rename(columns={"column": "feature"})
feature_register.insert(1, "group", feature_register["feature"].map(feature_to_group).fillna("unassigned"))
feature_register.insert(2, "review_template", feature_register["feature"].map(classify_feature))
feature_register["reviewed"] = False
feature_register["finding"] = ""
feature_register["baseline_treatment_to_test"] = ""
feature_register

In [ ]:
def inspect_feature(column: str, top_n: int = 15) -> None:
    """Show consistent univariate and target-aware diagnostics for one feature."""
    if column not in train_values.columns:
        raise KeyError(f"Unknown feature: {column}")

    series = train[column]
    review_template = classify_feature(column)
    summary = pd.Series(
        {
            "review_template": review_template,
            "dtype": str(series.dtype),
            "rows": len(series),
            "missing": int(series.isna().sum()),
            "missing_pct": series.isna().mean() * 100,
            "unique": int(series.nunique(dropna=True)),
        },
        name=column,
    )
    display(summary.to_frame())

    if review_template == "date":
        parsed = pd.to_datetime(series, errors="coerce")
        date_summary = pd.Series(
            {
                "earliest": parsed.min(),
                "latest": parsed.max(),
                "unparseable": int(parsed.isna().sum()),
                "distinct dates": int(parsed.nunique()),
            },
            name=column,
        )
        display(date_summary.to_frame())
        parsed.dt.to_period("M").value_counts().sort_index().plot(figsize=(10, 3), color="#376996")
        plt.title(f"{column}: records by month")
        plt.xlabel("Month")
        plt.ylabel("Rows")
    elif review_template == "identifier/free text":
        display(series.value_counts(dropna=False).head(top_n).to_frame("rows"))
        identifier_checks = pd.Series(
            {
                "unique_share_pct": series.nunique(dropna=True) / len(series) * 100,
                "values_seen_once": int((series.value_counts(dropna=False) == 1).sum()),
                "duplicated_non_null_values": int(series.dropna().duplicated().sum()),
            },
            name=column,
        )
        display(identifier_checks.to_frame())
        print("No target plot: first decide whether this feature is an identifier or usable text.")
        return
    elif review_template == "numeric":
        display(series.describe().to_frame(name=column).T)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        series.plot.hist(bins=40, ax=axes[0], color="#376996")
        axes[0].set(title=f"{column}: distribution", xlabel=column)
        train.boxplot(column=column, by="status_group", ax=axes[1], rot=20)
        axes[1].set(title=f"{column} by target", xlabel="", ylabel=column)
        fig.suptitle("")
    else:  # categorical or binary
        display(series.value_counts(dropna=False).head(top_n).to_frame("rows"))
        plot_values = series.astype("string").fillna("<missing>")
        top_categories = plot_values.value_counts().head(top_n).index
        comparison = pd.crosstab(
            plot_values,
            train["status_group"],
            normalize="index",
        ).loc[top_categories]
        comparison.plot.barh(stacked=True, figsize=(9, max(4, top_n * 0.3)))
        plt.title(f"{column}: target share within top categories")
        plt.xlabel("Share within category")
        plt.ylabel("")
        plt.legend(title="status_group", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    plt.show()


FEATURE_TO_REVIEW = "amount_tsh"
inspect_feature(FEATURE_TO_REVIEW)

### Suggested review order

Work through coherent groups instead of following the CSV order:

1. identifiers and free text;
2. target, money, capacity and time;
3. geography;
4. organisations, permissions and management;
5. extraction, payment, water, source and waterpoint hierarchies.

This order surfaces identifier leakage, missing-value conventions and duplicated concepts before they reach the preprocessing plan. Save only plots that support a decision.

## Course-first boundary

The first modelling pass will make the taught techniques visible and traceable:

| Course technique | Planned use |
| --- | --- |
| Descriptive statistics and visualisation | This audit and the feature register |
| Train/test splitting | A reproducible stratified validation split in notebook 02 |
| Decision trees | First interpretable model after a simple majority baseline |
| Oversampling | A controlled experiment on training folds only |
| K-fold cross-validation | Compare candidate models and reduce dependence on one split |
| Hyperparameter tuning | Tune the strongest taught model after establishing the baseline |

More specialised methods can follow once this sequence establishes a credible reference result. Each addition must answer a problem found in the audit and show a repeatable validation gain.

## Target balance

Overall accuracy can hide poor performance on `functional needs repair`, so measure the class balance before choosing validation metrics or resampling.

In [ ]:
target_counts = train_labels["status_group"].value_counts()
target_summary = pd.DataFrame(
    {
        "rows": target_counts,
        "share_pct": target_counts / target_counts.sum() * 100,
    }
)
display(target_summary.round(2))

ax = target_counts.sort_values().plot.barh(figsize=(8, 3.5), color="#376996")
ax.set(title="Training target balance", xlabel="Rows", ylabel="")
plt.tight_layout()
plt.show()

## Explicit nulls and possible categorical sentinels

Treat the tokens below as flags for review. Some uses of `none` may describe a real category rather than missing data.

In [ ]:
explicit_missing = inventory.loc[inventory["missing"] > 0, ["dtype", "missing", "missing_pct"]]
explicit_missing.sort_values("missing_pct", ascending=False).round(2)

In [ ]:
placeholder_tokens = {"", "none", "unknown", "not known", "n/a", "na", "missing", "-", "0"}
placeholder_rows = []

for column in train_values.select_dtypes(include=["object", "string"]).columns:
    normalised = train_values[column].astype("string").str.strip().str.lower()
    counts = normalised[normalised.isin(placeholder_tokens)].value_counts()
    for token, count in counts.items():
        placeholder_rows.append(
            {
                "column": column,
                "token": token,
                "rows": int(count),
                "share_pct": count / len(train_values) * 100,
            }
        )

categorical_placeholders = pd.DataFrame(placeholder_rows)
if categorical_placeholders.empty:
    display(categorical_placeholders)
else:
    display(categorical_placeholders.sort_values("rows", ascending=False).round(2))

## Numeric zeros and ranges

Zero has a plausible meaning for some columns and looks like a sentinel in others. Compare counts and ranges before deciding which zeros should become missing values.

In [ ]:
numeric_columns = train_values.select_dtypes(include=np.number).columns
numeric_summary = train_values[numeric_columns].describe().T
numeric_summary["zeros"] = (train_values[numeric_columns] == 0).sum()
numeric_summary["zero_pct"] = numeric_summary["zeros"] / len(train_values) * 100
numeric_summary.round(2)

In [ ]:
sentinel_candidates = [
    "amount_tsh",
    "gps_height",
    "longitude",
    "latitude",
    "num_private",
    "population",
    "construction_year",
]

zero_comparison = pd.DataFrame(
    {
        "train_zeros": (train_values[sentinel_candidates] == 0).sum(),
        "train_zero_pct": (train_values[sentinel_candidates] == 0).mean() * 100,
        "test_zeros": (test_values[sentinel_candidates] == 0).sum(),
        "test_zero_pct": (test_values[sentinel_candidates] == 0).mean() * 100,
    }
)
zero_comparison.round(2)

## Dates and construction year

Parse `date_recorded` without changing the source frame. Check whether it supports features such as recording year, pump age or time since construction.

In [ ]:
recorded_date = pd.to_datetime(train_values["date_recorded"], errors="coerce")
valid_construction_year = train_values["construction_year"].replace(0, np.nan)
pump_age = recorded_date.dt.year - valid_construction_year

date_checks = pd.Series(
    {
        "earliest date recorded": recorded_date.min(),
        "latest date recorded": recorded_date.max(),
        "unparseable dates": int(recorded_date.isna().sum()),
        "zero construction years": int((train_values["construction_year"] == 0).sum()),
        "earliest non-zero construction year": valid_construction_year.min(),
        "latest construction year": valid_construction_year.max(),
        "negative derived pump ages": int((pump_age < 0).sum()),
    },
    name="value",
)
date_checks.to_frame()

## Geographic coverage

A coordinate plot can expose sentinel coordinates, isolated points and regional coverage. Remove `(0, 0)` from the plot only; do not change the source data.

In [ ]:
coordinate_checks = pd.Series(
    {
        "rows at (0, 0)": int(((train["longitude"] == 0) & (train["latitude"] == 0)).sum()),
        "longitude minimum": train["longitude"].min(),
        "longitude maximum": train["longitude"].max(),
        "latitude minimum": train["latitude"].min(),
        "latitude maximum": train["latitude"].max(),
    },
    name="value",
)
display(coordinate_checks.to_frame())

plot_data = train.loc[(train["longitude"] != 0) & (train["latitude"] != 0)]
colours = {
    "functional": "#2a9d8f",
    "functional needs repair": "#e9c46a",
    "non functional": "#e76f51",
}

fig, ax = plt.subplots(figsize=(8, 7))
for label, group in plot_data.groupby("status_group"):
    ax.scatter(
        group["longitude"],
        group["latitude"],
        s=5,
        alpha=0.25,
        color=colours[label],
        label=label,
    )
ax.set(title="Training pumps with non-zero coordinates", xlabel="Longitude", ylabel="Latitude")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

## Related categorical fields

Several columns describe the same concept at different levels. Check whether one value maps cleanly to its parent category before deciding which versions to retain.

In [ ]:
related_pairs = [
    ("extraction_type", "extraction_type_group"),
    ("extraction_type_group", "extraction_type_class"),
    ("management", "management_group"),
    ("payment", "payment_type"),
    ("water_quality", "quality_group"),
    ("quantity", "quantity_group"),
    ("source", "source_type"),
    ("source_type", "source_class"),
    ("waterpoint_type", "waterpoint_type_group"),
]

mapping_rows = []
for child, parent in related_pairs:
    child_to_multiple_parents = (train_values.groupby(child, dropna=False)[parent].nunique() > 1).sum()
    mapping_rows.append(
        {
            "child": child,
            "parent": parent,
            "child_categories": train_values[child].nunique(dropna=False),
            "parent_categories": train_values[parent].nunique(dropna=False),
            "children_mapping_to_multiple_parents": int(child_to_multiple_parents),
        }
    )

category_mappings = pd.DataFrame(mapping_rows)
category_mappings

## Train/test category drift

Unseen test categories will break a naive encoder. Count them now so the preprocessing pipeline can handle unknown values.

In [ ]:
drift_rows = []
categorical_columns = train_values.select_dtypes(include=["object", "string"]).columns

for column in categorical_columns:
    train_categories = set(train_values[column].dropna().astype(str))
    test_as_text = test_values[column].dropna().astype(str)
    unseen = set(test_as_text) - train_categories
    drift_rows.append(
        {
            "column": column,
            "train_categories": len(train_categories),
            "unseen_test_categories": len(unseen),
            "test_rows_with_unseen_category": int(test_as_text.isin(unseen).sum()),
        }
    )

category_drift = pd.DataFrame(drift_rows).set_index("column")
category_drift.sort_values(
    ["test_rows_with_unseen_category", "unseen_test_categories"],
    ascending=False,
).head(20)

## Decision log for the baseline

Use the measurements above to turn each issue into an explicit choice. The entries below are prompts, not settled preprocessing rules.

In [ ]:
decision_log = pd.DataFrame(
    [
        {
            "issue": "Minority target class",
            "evidence": f"Smallest class is {target_summary['share_pct'].min():.1f}% of training rows.",
            "decision_for_02": "Use a stratified split; compare accuracy with per-class recall and macro F1.",
        },
        {
            "issue": "Explicit missing values",
            "evidence": f"{len(explicit_missing)} columns contain explicit nulls.",
            "decision_for_02": "Choose numeric and categorical imputation inside the pipeline.",
        },
        {
            "issue": "Numeric sentinel values",
            "evidence": f"{int((train_values['construction_year'] == 0).sum()):,} rows have construction_year=0.",
            "decision_for_02": "Review each zero-heavy field; do not apply one rule to every numeric zero.",
        },
        {
            "issue": "Coordinate quality",
            "evidence": f"{int(((train_values['longitude'] == 0) & (train_values['latitude'] == 0)).sum()):,} rows use (0, 0).",
            "decision_for_02": "Represent missing coordinates and test whether geography improves validation.",
        },
        {
            "issue": "High-cardinality categories",
            "evidence": ", ".join(high_cardinality_columns) or "No columns crossed the 20% threshold.",
            "decision_for_02": "Exclude identifier-like text from the first baseline; revisit with grouped features.",
        },
        {
            "issue": "Unseen test categories",
            "evidence": f"{int((category_drift['unseen_test_categories'] > 0).sum())} categorical columns contain unseen test levels.",
            "decision_for_02": "Configure the encoder to ignore unknown categories.",
        },
        {
            "issue": "Related categorical hierarchies",
            "evidence": f"Audited {len(related_pairs)} child/parent pairs.",
            "decision_for_02": "Start with one useful level per concept, then compare against retaining all levels.",
        },
    ]
)

decision_log

## Before starting `02-baseline.ipynb`

Record the choices supported by this audit:

1. validation split and metrics;
2. columns to exclude from the first baseline;
3. values to treat as missing, column by column;
4. numeric and categorical imputation rules;
5. date and geographic features worth deriving;
6. how the encoder will handle rare and unseen categories.

Keep the first baseline plain. Later notebooks can test whether more elaborate treatment produces a repeatable validation gain.